# Árbol de clasificación — predicción de `opinion`

Obligatorio MLSI 2026 S2. Esta notebook contiene:

1. Carga de datos
2. Preprocesamiento (código común en `obl_utils.py`)
3. Validación y métricas
4. *(próximo)* Baseline, búsqueda de hiperparámetros y submission a Kaggle

## 1. Carga de datos

In [ ]:
import os
import urllib.request

# El código común (preprocesamiento y evaluación) está en obl_utils.py.
# En Colab no está el archivo, así que se descarga del repo.
if not os.path.exists('obl_utils.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/juanmareolon/OBL_ML/main/obl_utils.py', 'obl_utils.py')

import numpy as np
import pandas as pd
from obl_utils import *

train, test = cargar_datos()
X, y, X_test, ids_test = preparar_datos(train, test)

print('train:', train.shape, '| test:', test.shape)
print('X:', X.shape, '| X_test:', X_test.shape)
print('Balance de clases:', y.value_counts(normalize=True).round(3).to_dict())
X.head()

## 2. Preprocesamiento

El código está en `obl_utils.py` y es el mismo para todas las notebooks de modelos, así la comparación entre modelos es justa.

El preprocesamiento tiene dos capas:

**Capa 1: limpieza fija (`limpiar`).** Transformaciones que no aprenden nada de los datos, así que se pueden aplicar una sola vez a train y test sin riesgo de *data leakage*:

| Columna | Decisión | Motivo |
|---|---|---|
| `budget`, `revenue` | 0 y valores < 1000 → NaN, más indicador `*_desconocido` | Los 0 son faltantes (21% y 29% de las filas) y los valores < 1000 son errores de unidad (ej. *Split Second*: budget = 7). Que falte el dato puede ser informativo. |
| `runtime` | 0 → NaN | Ninguna película dura 0 minutos. |
| `vote_count` | Se mantienen los 0 | Es un dato real: las 45 películas con 0 votos tienen `opinion` = 0. |
| `budget`, `revenue`, `vote_count`, `popularity` | `log1p` | Distribuciones muy sesgadas (ej. `popularity`: mediana 13, máximo 875). |
| `release_date` | Se extraen `anio` y `mes` | El año está muy asociado a `opinion` (pre-1980: 79% positivas; post-2005: 45%) y el mes también (temporada de premios a fin de año). |
| `keywords` | `sin_keywords` y `cant_keywords` | Las películas sin keywords tienen 31% de opinión positiva contra 49% del total. |
| nueva | `log_roi` = log1p(revenue / budget) | Relación recaudación/costo; solo si ambos datos existen. |
| `status`, `original_title`, `overview` | Se descartan | `status` es 99,8% "Released", el título es casi un id y `overview` queda para la parte de NLP. |

**Capa 2: pipeline de sklearn (`crear_preprocesador`).** Todo lo que *aprende* de los datos (medianas, escalas, categorías frecuentes, encodings) va dentro de un `ColumnTransformer`. Así, en cada fold de la validación cruzada se ajusta solo con el train de ese fold.

- **Numéricas:** imputación por mediana + `StandardScaler` (necesario para la regresión logística/lineal; no afecta a los árboles).
- **`main_genre_top10`:** one-hot.
- **`original_language`:** one-hot, agrupando los idiomas con menos de 20 películas en una categoría "infrecuente" (quedan inglés, francés, español y otros).
- **`director`** (1942 valores, 1275 con una sola película) con tres variantes para comparar:
  - `'ninguno'`: no se usa.
  - `'freq'`: frequency encoding (cantidad de películas del director en el train).
  - `'freq+target'`: frequency + target encoding con suavizado (`TargetEncoder`, que usa validación cruzada interna para no filtrar el target).

## 3. Validación y métricas

**Validación: k-fold cross-validation estratificado** (`crear_cv()`), con 5 folds y `SEED = 42`. Todos los modelos usan exactamente los mismos folds.

- Cada película se usa para validar exactamente una vez, a diferencia del holdout (depende de una sola división al azar) y del repeated holdout (algunas películas se validan varias veces y otras ninguna).
- *Estratificado*: cada fold mantiene la proporción de clases (51/49).
- El preprocesador va dentro de un `Pipeline` junto al modelo (`crear_pipeline()`), así medianas, escalas y encodings se ajustan solo con el train de cada fold (sin *data leakage*).

**Métricas** (`evaluar()`): accuracy, precision, recall, F1 y AUC-ROC, con promedio y desvío entre folds. También se registra el AUC en train: si es mucho mayor que el de validación, el modelo está sobreajustando. La matriz de confusión se obtiene con `matriz_confusion()`, usando las predicciones de validación de cada fold.

La métrica principal para elegir es el **AUC**, porque es la que usa la competencia de Kaggle.

## 4. Árbol de clasificación

*(Próximo paso: baseline y búsqueda de hiperparámetros.)*